# MimicGuard: detector benchmark on Kaggle

Simulation-only study: an autoencoder trained on normal fiber-wireless channel data vs classical detectors
(energy, CUSUM, PCA residual, OC-SVM, Isolation Forest, LOF) against a channel-mimicking eavesdropper with
imitation fidelity alpha. Thresholds come from a 1% false-alarm rate on normal validation data; attack data is never
used for training or thresholds.

**How to run:** Settings -> Accelerator *None* (CPU is enough; the GPU gives no speed-up for this small model),
then *Run All*. Cells run in order: setup -> unit tests -> smoke test (~1 min) -> experiments (main sweep with temporal CUSUM-wrapped detectors, feature/latent ablations, AGC and window-length robustness checks; about 1-2 h on CPU, estimate) -> reports.
Set `MODE` in the config cell: `"smoke"` stops after the smoke test, `"medium"` = few seeds, `"full"` = frozen grid (10 seeds main, 5 seeds for ablations and robustness).
The sweep is resumable: if the session stops, run the cells again and finished cells are skipped (keep the same output folder).

In [ ]:
# ---- 1. Setup: unpack the embedded code ----
import base64, io, os, sys, zipfile, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("mimicguard_run")
os.makedirs(WORK, exist_ok=True)
REPO = os.path.join(WORK, "mimicguard")
_B64 = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_B64))) as z:
    z.extractall(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.chdir(REPO)
print("unpacked to", REPO, sorted(os.listdir(REPO)))

In [ ]:
# ---- 2. Environment ----
import platform, numpy, scipy, sklearn
print("python", platform.python_version(), "| numpy", numpy.__version__, "| scipy", scipy.__version__, "| sklearn", sklearn.__version__)
try:
    import torch
    print("torch", torch.__version__, "-> autoencoder backend: torch")
except Exception:
    print("torch missing -> autoencoder falls back to sklearn MLP (named ae_mlp in results)")
try:
    import yaml
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
print("CPUs:", os.cpu_count())

In [ ]:
# ---- 3. Unit tests (channel, attacker, features, detectors) ----
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-1500:])
assert r.returncode == 0, "unit tests failed: stop and report the output above"

In [ ]:
# ---- 4. Config ----
MODE = "full"          # "smoke" | "medium" | "full"
WORKERS = os.cpu_count() or 1
OUT = os.path.join(WORK, "results")          # keep fixed so a re-run resumes
os.makedirs(OUT, exist_ok=True)
def sweep(name, *extra):
    cmd = [sys.executable, "-u", "experiments/run_sweep.py", "--out", os.path.join(OUT, name),
           "--workers", str(WORKERS), *extra]
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONWARNINGS": "ignore"})
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="")
    assert p.wait() == 0, "sweep failed"

In [ ]:
# ---- 5. Smoke test: 8 cells, 20 realisations each ----
sweep("smoke", "--quick")
from mimicguard.report import load_results
import pandas as pd; pd.set_option("display.width", 200)
d = load_results(os.path.join(OUT, "smoke"))
ref = d[(d.fading == "none") & (d.snr_db == 20)]
print("AUC at no fading, 20 dB (rows: detector, cols: alpha)")
print(ref.pivot_table(index="detector", columns="alpha", values="auc").round(3))
print("mean empirical false-alarm rate on test (target 0.01):")
print(d.groupby("detector").far_test.mean().round(3))

In [ ]:
# ---- 6. Experiments ----
# main: frozen grid, all base detectors plus temporal (CUSUM-wrapped) versions
CORE = "random,energy,cusum,pca,ocsvm,iforest,lof,ae"
SEQ = "pca_cusum,ocsvm_cusum,iforest_cusum,lof_cusum,ae_cusum"
FEATS = ["power", "evm", "pilot_mag", "pilot_dphase", "amp_var", "amp_kurt"]
ABL = ",".join(["ae", "ae_L2", "ae_L4", "ae_L6", "pca", "pca_L2", "pca_L4"]
               + [f"ae_drop_{f}" for f in FEATS] + [f"pca_drop_{f}" for f in FEATS])
ROB = "energy,cusum,pca,lof,ae,pca_cusum,ae_cusum"
if MODE == "smoke":
    print("MODE=smoke: skipping experiments")
else:
    S = ["--seeds", "0-2"] if MODE == "medium" else []
    sweep("main", "--detectors", CORE + "," + SEQ, *S)
    S5 = ["--seeds", "0-1"] if MODE == "medium" else ["--seeds", "0-4"]
    sweep("ablation", "--detectors", ABL, "--snr", "10,20,30", *S5)
    for tag, ov in [("rob_agc", '{"handover":{"mode":"agc"}}'),
                    ("rob_win128", '{"signal":{"window_symbols":128}}'),
                    ("rob_win512", '{"signal":{"window_symbols":512}}')]:
        sweep(tag, "--detectors", ROB, "--snr", "5,10,20,30", "--override", ov, *S5)

In [ ]:
# ---- 7. Report: main sweep ----
from mimicguard.report import load_results, summarize, paired_test, plot_all, table, compare_seq
from IPython.display import Image, display
import pandas as pd; pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
name = "smoke" if MODE == "smoke" else "main"
rd = os.path.join(OUT, name)
df = load_results(rd)
print(len(df), "rows;", df.seed.nunique(), "seeds;", df.detector.nunique(), "detectors")
for f in sorted(df.fading.unique()):
    for snr in (20, 5):
        for m in ("auc", "pd"):
            print(f"=== {m}, fading={f}, SNR {snr} dB (mean over seeds) ===")
            print(table(df, m, f, snr))
        print()
print("=== median detection delay (windows after onset, detected streams only), no fading, 20 dB ===")
print(table(df, "delay", "none", 20, digits=1))
print("=== share of streams alarmed after onset, no fading, 5 dB ===")
print(table(df, "det_frac", "none", 5))
summarize(df).to_csv(os.path.join(rd, "summary_by_condition.csv"))
df.to_csv(os.path.join(rd, "all_rows.csv"), index=False)
pt = paired_test(df); pt.to_csv(os.path.join(rd, "paired_ae_vs_others.csv"), index=False)
cs = compare_seq(df)
if len(cs):
    print("=== gain in AUC from adding CUSUM to each detector (alpha 0.25-0.75), by fading and SNR ===")
    print(cs)
    cs.to_csv(os.path.join(rd, "seq_gain.csv"))
for p in plot_all(df, rd):
    display(Image(filename=str(p)))

In [ ]:
# ---- 7b. Report: ablations and robustness ----
if MODE != "smoke":
    ab = load_results(os.path.join(OUT, "ablation"))
    ab.to_csv(os.path.join(OUT, "ablation", "all_rows.csv"), index=False)
    for f in sorted(ab.fading.unique()):
        print(f"=== ablation AUC, fading={f}, SNR 20 dB ===")
        print(table(ab, "auc", f, 20))
    for tag in ("rob_agc", "rob_win128", "rob_win512"):
        rb = load_results(os.path.join(OUT, tag))
        rb.to_csv(os.path.join(OUT, tag, "all_rows.csv"), index=False)
        for f in ("none", "rayleigh"):
            print(f"=== {tag}: AUC, fading={f}, SNR 20 dB ===")
            print(table(rb, "auc", f, 20))

In [ ]:
# ---- 8. Package the outputs for download ----
import shutil
zp = shutil.make_archive(os.path.join(WORK, "mimicguard_results"), "zip", OUT)
print("results archive:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)")
print("On Kaggle: Save Version -> Save & Run All (Commit); files in /kaggle/working appear under Output.")